# Data Quality Toolkit — quick demo

This notebook shows the whole workflow: check a messy dataset, clean it up, and generate a shareable report. Every cell here runs as-is, using a small made-up dataset, so you can run the whole thing top to bottom with no setup beyond the install cell below.

At the end there's a short section on swapping in your own CSV or Excel file instead.

Full function reference: see the README in the repo.

In [ ]:
# install the library straight from GitHub
!pip install git+https://github.com/Yanaantonyuk/data-quality-passport.git -q

## 1. A messy dataset to work with

This has a bit of everything: a missing value, a disguised missing value ("ERROR"), inconsistent name casing, an outlier, and a couple of date problems.

In [ ]:
import pandas as pd
from data_quality_toolkit import (
    quality_report, fill_disguised_missing, fill_missing,
    standardise_casing, standardise_dates, flag_outliers,
    ai_summary, generate_dashboard
)

df = pd.DataFrame({
    'Order ID': range(1, 13),
    'Customer': ['Anna', 'anna', 'Boris', 'Boris', 'Carla', 'UNKNOWN', 'Dmitri', 'Dmitri', 'Elena', 'elena', 'Farid', 'Farid'],
    'Amount': [45.0, 32.0, None, 28.0, 51.0, 39.0, 'ERROR', 44.0, 3200.0, 47.0, 29.0, 33.0],
    'Order Date': ['2024-01-05', '2024-01-06', '05/01/2024', '2024-01-08', '2024-01-09',
                   '2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', 'not a date',
                   '2024-01-15', '2024-01-16']
})

df

## 2. Run the diagnosis

`quality_report()` checks everything at once. On a dataset this small, the automatic column detection can guess wrong — here it doesn't pick "Customer" as a text-consistency column on its own, so we tell it to check that one explicitly. That's the same override pattern you would use on a real dataset if a check ever guesses the wrong column.

In [ ]:
report = quality_report(df, text_consistency_columns=['Customer'])

print('Amount missing:', report['columns']['Amount']['missing_percent'], '%')
print('Customer casing issues:', report['text_consistency']['Customer'])
print('Outliers in Amount:', report['outliers']['Amount'])
print('Date problems:', report['date_checks']['Order Date'])

Notice the date check flags 3 different formats in that column, including one that failed to parse entirely. Automatic date parsing can guess wrong on ambiguous formats like `05/01/2024` (5th of January, or May 1st?) — this is exactly the kind of thing worth checking by eye rather than trusting blindly, which is why the report surfaces it instead of silently fixing it.

## 3. Clean it up

Each function does one thing and returns a new DataFrame, so you can chain them in whatever order makes sense, or stop after any step and inspect the result.

In [ ]:
df = fill_disguised_missing(df)                 # "UNKNOWN" / "ERROR" -> real NaN
df = fill_missing(df, 'Amount', 'median')        # fill the real gap
df = standardise_casing(df, 'Customer')          # "Anna" / "anna" -> one spelling
df = standardise_dates(df, 'Order Date')         # consistent YYYY-MM-DD
df = flag_outliers(df, 'Amount')                 # flags, does not remove

df

Note the 3200.0 order is flagged as an outlier but is still sitting right there in the data — nothing was deleted. Whether that's a genuine large order or a data entry error is a judgement call, so that's left to you rather than decided automatically.

## 4. Share the results

If you want to hand this to someone who isn't going to read a DataFrame, turn it into a plain-language summary and an HTML dashboard.

In [ ]:
summary_text = ai_summary(report)  # needs an ANTHROPIC_API_KEY env var, otherwise a simple fallback summary is used instead
generate_dashboard(report, summary_text, 'report.html')

print(summary_text)

In Colab, download the dashboard from the file browser on the left (the folder icon), or run this to grab it directly:

In [ ]:
from google.colab import files
files.download('report.html')

## Using your own file instead

Swap the sample DataFrame above for:

```python
from data_quality_toolkit import load_dataset

df = load_dataset('your_file.csv')                       # or .xlsx
df = load_dataset('your_file.xlsx', sheet_name='Orders')  # for a specific sheet
```

If you're in Colab and the file is on your computer, upload it first:

```python
from google.colab import files
uploaded = files.upload()
```

Then run the same `quality_report()` → clean → `generate_dashboard()` steps above on your own data.